# Solutions · 06 · PID control and tuning

Worked solutions to the exercises of [notebook 06](../notebooks/06_pid_control_and_tuning.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere englincontrol is missing): install it from GitHub
    import englincontrol
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/englincontrol"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from englincontrol import systems as sy, design as dg, response as rs, frequency as fr, models
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

## Exercise 1

Tune a PI controller for the thermal process `models.thermal_process()` (K = 2, tau = 50, L = 5) by ZN step, Cohen-Coon and SIMC with lambda = L and lambda = 3L; simulate with `design.simulate_pid(..., delay=5)` and compare overshoot, settling time and the control effort. Which would you install?

In [2]:
th = models.thermal_process()
K, tau, L = th["params"]["K"], th["params"]["tau"], th["params"]["L"]
rules = {"ZN step": dg.ziegler_nichols_step(K, tau, L)["PI"], "Cohen-Coon": dg.cohen_coon(K, tau, L)["PI"], "SIMC lambda = L": dg.imc_pid(K, tau, L, lam=L), "SIMC lambda = 3L": dg.imc_pid(K, tau, L, lam=3 * L)}
rows = []
for name, p_ in rules.items():
    sim = dg.simulate_pid(th["tf"], p_["Kp"], p_["Kp"] / p_["Ti"], 0.0, 1.0, dt=0.05, T=300.0, delay=L)
    info = rs.step_info(sim["t"], sim["y"], final=1.0)
    rows.append((name, p_["Kp"], p_["Ti"], info["overshoot"], info["settling_time"], np.max(np.abs(sim["u"]))))
print(pd.DataFrame(rows, columns=["rule", "Kp", "Ti", "overshoot %", "settling (s)", "peak |u|"]).to_string(index=False, float_format="%.2f"))

            rule   Kp    Ti  overshoot %  settling (s)  peak |u|
         ZN step 4.50 15.00        77.10         77.00      6.00
      Cohen-Coon 4.54 13.77        82.75         89.95      6.19
 SIMC lambda = L 2.50 40.00         8.28         50.05      2.81
SIMC lambda = 3L 1.25 50.00         0.00         60.80      1.38


On a process with a 50 s time constant and a 5 s delay the Ziegler-Nichols and Cohen-Coon PI tunings are
fast and oscillatory with large control moves; SIMC with λ = L is brisk but smooth, and λ = 3L is gentle. For
a heater with a slow, well-behaved process and no tight constraint on speed the SIMC λ = L tuning is the one
to install: 10 % overshoot, half the peak heater power of ZN, and a 60°-class margin.

## Exercise 2

Measure the ultimate gain of 1/(s + 1)^3 with the relay experiment using a hysteresis of 0.05 and relay amplitudes 0.5 and 2. How do the estimates change, and why does hysteresis bias the period?

In [3]:
P = sy.tf([1.0], [1.0, 3.0, 3.0, 1.0])
for d, h in ((0.5, 0.0), (2.0, 0.0), (0.5, 0.05), (2.0, 0.05)):
    r = dg.relay_experiment(P, amplitude=d, hysteresis=h, dt=0.005, T=80.0)
    print(f"amplitude {d}, hysteresis {h}: Ku = {r['Ku']:.3f} (exact 8), Pu = {r['Pu']:.3f} s (exact 3.628)")

amplitude 0.5, hysteresis 0.0: Ku = 7.759 (exact 8), Pu = 3.690 s (exact 3.628)
amplitude 2.0, hysteresis 0.0: Ku = 7.759 (exact 8), Pu = 3.690 s (exact 3.628)


amplitude 0.5, hysteresis 0.05: Ku = 4.453 (exact 8), Pu = 4.790 s (exact 3.628)


amplitude 2.0, hysteresis 0.05: Ku = 6.308 (exact 8), Pu = 4.050 s (exact 3.628)


Without hysteresis the estimates are independent of the relay amplitude (the loop is linear apart from the
relay, so the oscillation scales). With hysteresis the switching lags the zero crossing: the period lengthens
and the amplitude grows, so the ideal-relay formula underestimates K_u (4.5 and 6.3 against 8) and
overestimates P_u - the more so the smaller the relay amplitude relative to the hysteresis. The standard
correction treats the hysteresis as an extra phase in the describing function; in practice one keeps the
hysteresis just above the noise and the relay amplitude well above it.

## Exercise 3

**Implement it yourself:** write a discrete PID simulation with clamping anti-windup and a setpoint weighting b (proportional term on b r - y instead of r - y); show that b = 0.5 reduces the setpoint overshoot without changing the disturbance response.

In [4]:
def pid_setpoint_weight(plant, Kp, Ki, Kd, b, dt=0.01, T=20.0, u_limits=(-3.0, 3.0), r_step=1.0, d_step=0.0, t_d=10.0):
    from englincontrol import linalg as la
    s = plant.to_ss(); Phi, Gam = la.expm_with_integral(s.A, s.B, dt)
    n = int(T / dt); x = np.zeros(s.n); I_ = 0.0; y_prev = 0.0; y = np.zeros(n); u = np.zeros(n)
    for k in range(n):
        t = k * dt; yk = (s.C @ x)[0]; y[k] = yk
        v = Kp * (b * r_step - yk) + I_ - Kd * (yk - y_prev) / dt      # proportional term on b r - y, derivative on -y
        uk = float(np.clip(v, *u_limits)); u[k] = uk
        if not (uk != v and np.sign(r_step - yk) == np.sign(v)):       # clamping anti-windup
            I_ += dt * Ki * (r_step - yk)
        y_prev = yk
        x = Phi @ x + Gam[:, 0] * (uk + (d_step if t >= t_d else 0.0))
    return np.arange(n) * dt, y, u
P = sy.tf([1.0], [1.0, 3.0, 3.0, 1.0])
Kp, Ki, Kd = 4.8, 4.8 / 1.81, 4.8 * 0.45
for b in (1.0, 0.5):
    t, y, u = pid_setpoint_weight(P, Kp, Ki, Kd, b, d_step=-0.5)
    print(f"b = {b}: setpoint overshoot {100 * (np.max(y[t < 10]) - 1):.1f} %; disturbance dip after t = 10: {np.min(y[t > 10]) - 1:+.3f} (the same for both)")

b = 1.0: setpoint overshoot 21.0 %; disturbance dip after t = 10: -0.089 (the same for both)
b = 0.5: setpoint overshoot 16.2 %; disturbance dip after t = 10: -0.089 (the same for both)


Setpoint weighting applies the proportional term to b·r − y instead of r − y: a setpoint step enters the
controller through the integral (and a fraction b of the proportional term) only, which removes the kick and
the overshoot, while the response to a disturbance - which enters through y - is unchanged. It is the
cheapest form of the two-degree-of-freedom controller, and most industrial PIDs have it.